In [ ]:
# intraday.py
# 抓 intraday（yfinance），做基本檢查，並支援 fetch_until 過濾
import yfinance as yf
import pandas as pd
from database import get_engine
import time as _time

def fetch_intraday_safe(symbol: str, interval: str, period: str, fetch_until=None):
    """
    抓 intraday，做基本檢查：
      - df not None
      - df not empty
      - 必要欄位存在且 dropna 後仍有 rows
    回傳 DataFrame 或 None（欄位：symbol, ts, open, high, low, close, volume）
    """
    try:
        df = yf.download(symbol, interval=interval, period=period, progress=False)
    except Exception as e:
        print(f"fetch error {symbol} {interval}: {e}")
        return None

    if df is None or df.empty:
        return None

    required = ["Open","High","Low","Close"]
    if not all(c in df.columns for c in required):
        return None

    df = df.dropna(subset=required)
    if df.empty:
        return None

    df = df.reset_index()
    dt_col = df.columns[0]
    if fetch_until is not None:
        df = df[df[dt_col] <= fetch_until]
        if df.empty:
            return None

    df = df.rename(columns={dt_col: "ts", "Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
    df["symbol"] = symbol
    return df[["symbol","ts","open","high","low","close","volume"]]

def save_intraday(df, data_type):
    """把 df 寫入主表 stock_data（簡單示範）"""
    if df is None or df.empty:
        return 0
    engine = get_engine()
    df = df.copy()
    df["data_type"] = data_type
    df2 = df[["symbol","data_type","ts","open","high","low","close","volume"]]
    try:
        # 簡單用 pandas.to_sql；若要避免重複建議改用 INSERT ... ON CONFLICT
        df2.to_sql("stock_data", engine, if_exists="append", index=False)
        return len(df2)
    except Exception as e:
        print("save_intraday error:", e)
        return 0

def update_intraday_for_symbols(symbols, interval, fetch_until=None, per_symbol_sleep=0.6):
    """抓多個 symbols 的 intraday 並儲存"""
    period = "7d" if interval == "1m" else "60d"
    total = 0
    for s in symbols:
        df = fetch_intraday_safe(s, interval, period, fetch_until=fetch_until)
        if df is not None:
            n = save_intraday(df, interval)
            total += n
        _time.sleep(per_symbol_sleep)
    return total

if __name__ == "__main__":
    print(update_intraday_for_symbols(["AAPL"], "1m"))
